In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from config import DATA_PATH


In [ ]:
fnames_probs = {
    '2024b': '2024BehaviorAgentProbabilities.csv',
    # '2025a': '2025AllAgentProbabilities.csv',
    # '2025b': '2025BehaviorAgentProbabilities.csv'
}
fnames_dfs = {
    '2024': 'dfHourly2024.csv',
    # '2025': 'dfHourly2025.csv'
}
probs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_probs.items()}
dfs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_dfs.items()}
# msgs = pd.read_csv(os.path.join(DATA_PATH, 'IHSMessagePool.csv'))

In [ ]:
for k in probs:
    print(f"Shape of prob {k}: {probs[k].shape}")
    print(f"Columns of {k}: {probs[k].columns[~probs[k].isna().all(axis=0)]}")
for k in dfs:
    print(f"Shape of df {k}: {dfs[k].shape}")
    print(f"Columns of {k}: {dfs[k].columns[~dfs[k].isna().all(axis=0)]}")
print(f"Shape of messages: {msgs.shape}")
print(f"Columns of messages: {msgs.columns[~msgs.isna().all(axis=0)]}")

# 2024

## Variables Needed
* Contextual variables 
* Intervention & Probability
    * Send vs no send: `action_id` = {1: 'Send', 0: 'No Send'}
    * Domain: `message_metric`
    * Chosen message: `core_message_id`
    * Message strategy: `strategy`
    * uniform for send vs no send: `p_send`
    * uniform for strategy & domain of chosen message (without replacement!): `p_mood`, `p_steps`, `p_sleep`
        * these are conditional probabilities P(Domain|Action=1)
* BL Characteristics

### Intervention & Probability

In [ ]:
df = probs['2024b'].copy()

# convert to datetime
df['recorded_time_utc'] = pd.to_datetime(df['recorded_time_utc'], utc=True)
df['decision_date'] = pd.to_datetime(df['decision_date'])

# verify decision_date reflects time in UTC
assert (df['decision_date'] == df['recorded_time_utc'].dt.date).all()

# verify Preinternship notification => before July 1
p = df['notification_id'].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0] == 'Preinternship'
q = df['recorded_time_utc'] < pd.Timestamp("2024-07-01", tz="UTC")
assert (~p | q).all()
# verify after July 1 => not Preinternship notification
assert (q | ~p).all()

# verify NoMessage notification <=> action_id == 0
p = df['notification_id'].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0] == 'NoMessage'
q = df['action_id'] == 0
assert (~p | q).all()
assert (~q | p).all()

# filter out preinternship
df = df.loc[df['recorded_time_utc'] >= pd.Timestamp("2024-07-01", tz="UTC")]

# verify action_id == 1 <=> core_message_id
assert (~(df['action_id'] == 1) | (~df['core_message_id'].isna())).all()
assert (df['core_message_id'].isna() | (df['action_id'] == 1)).all()

# verify probability is not missing
assert df[['p_send', 'p_mood', 'p_steps', 'p_sleep']].notnull().all().all()

# verify there is only 1 notification per day per participant
s = df\
    .groupby('participant_id')['decision_date']\
    .apply(lambda s: s.duplicated().sum())
assert (s == 0).all()

# verify no send <=> missing metric & missing strategy
p = df['action_id'] == 0
q = df['message_metric'].isnull() & df['strategy'].isnull()
assert (~p | q).all()
assert (~q | p).all()

# verify chosen message has consistent domain and strategy
assert df\
    .groupby('core_message_id')[['message_metric', 'strategy']]\
    .nunique().eq(1).all(axis=1).all()

# TODO: verify domain probabilities add up to one
assert np.isclose(df[['p_mood', 'p_steps', 'p_sleep']].sum(axis=1), 1).all()

# subset to relevant columns
df = df[[
    'decision_id',
    'participant_id',
    'decision_date',
    'action_id',
    'message_metric',
    'strategy',
    'p_send',
    'p_mood',
    'p_steps',
    'p_sleep'
]].set_index('decision_id')

# export
df.to_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))

### Contextual Variables & Outcomes

In [ ]:
df = dfs['2024'].copy()

# get date
df['decision_date'] = pd.to_datetime(df['time_local']).dt.date

# TODO: implement loop for each device
# subset to fitbit
idx = ['participant_id', 'decision_date', 'time_local']
cols = df.columns[df.columns.str.startswith('Fitbit_')]
df = df.loc[df[cols].notna().any(axis=1), [*idx, *cols, 'Mood']].set_index(idx)

# TODO: aggregate
df = df.groupby(['participant_id', 'decision_date']).mean()

# export
df.to_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))

## Combine

In [ ]:
df1 = pd.read_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))\
    .set_index(['participant_id', 'decision_date'])
df2 = pd.read_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))\
    .set_index(['participant_id', 'decision_date'])
df = df1.merge(df2, how='outer', left_index=True, right_index=True, indicator=True)

In [ ]:
df.reset_index().groupby('_merge')['participant_id'].count()